# Step 2 - Does the training loop run?

Before we put the real reward in, we check the training machinery runs at all.
This trains for just a few steps with a FAKE reward (a random number). We are
not looking for the model to get better. We only want it to run start to finish
without crashing.

If this works, the risky part of the project is behind us.

## Setup: get the repo

In [1]:
import os, sys, subprocess

REPO = "https://github.com/ookino/rlvr-argument-mining.git"
NAME = "rlvr-argument-mining"

# Make sure we are inside the repo folder.
if os.path.basename(os.getcwd()) != NAME:
    if not os.path.isdir(NAME):
        subprocess.run(["git", "clone", REPO], check=True)
    os.chdir(NAME)

# Always pull the latest code so the runtime is never stale.
subprocess.run(["git", "pull", "--quiet"], check=False)

sys.path.insert(0, os.getcwd())
print("repo ready, at", os.getcwd())

repo ready


## Install the training libraries

`unsloth` loads the model cheaply in 4-bit and gives us fast GRPO. It pulls in
`trl` (which has the GRPO trainer), `transformers`, `peft` and the rest. This
cell takes a few minutes the first time.

In [2]:
!pip install -q unsloth

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.1/76.1 MB 35.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 68.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 46.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 158.9 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 38.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 87.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 130.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 134.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.0/215.0 kB 24.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 56.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 129.2 MB/s eta 0:00:00
   ━━━━━━━━

## Print the library versions

If the training cell later fails, these versions are the first thing to check,
because the GRPO library changes its function names between versions.

In [3]:
import unsloth          # import first so it can patch the others
import torch, transformers, trl
print("unsloth     ", unsloth.__version__)
print("trl         ", trl.__version__)
print("transformers", transformers.__version__)
print("torch       ", torch.__version__)
print("gpu         ", torch.cuda.is_available())

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
unsloth      2026.7.5
trl          0.24.0
transformers 5.5.0
torch        2.11.0+cu128
gpu          True


## Run the smoke test

This loads Qwen 2.5 3B in 4-bit, adds the small trainable adapters, and runs 5
GRPO steps on a handful of toy questions with a random reward. Loading the model
takes a minute. The 5 steps are slow because the model writes several answers
per question.

Success looks like a short training table and the line:

    training loop finished without crashing

In [4]:
from train.grpo_train import run

trainer = run("configs/baseline.yaml", max_steps=5)

ModuleNotFoundError: No module named 'train.grpo_train'

## What to do next

- If it printed **training loop finished without crashing**: step 2 is done.
  Tell Claude and we wire in the real reward.
- If it **errored**: copy the whole error and paste it back. Version mismatches
  in the GRPO library are common and quick to fix once we see the message.